In [1]:
import pandas as pd
import os
import glob
from pathlib import Path

In [ ]:
def cargar_csv():
    """
    Carga los archivos principales de la EPA y los anexos
    desde data/raw y devuelve dos diccionarios:
    - dfs: archivos principales
    - dfs_a: anexos
    """

    ruta = Path("../data/raw")  # carpeta donde están los datos

    dfs = {}    # aquí guardamos los df principales
    dfs_a = {}  # aquí guardamos los df de los anexos

    # Carga de archivos principales

    for file in sorted(ruta.glob("EPA_*")):
        if file.suffix == ".tab":
            df = pd.read_csv(file, sep="\t", encoding="utf-8", low_memory=False)
        else:
            df = pd.read_csv(file, sep="\t", encoding="utf-8", low_memory=False)
            # esto lo dejamos así porque, aunque la extensión sea distinta,
            # en la práctica también van separados por tabulación

        key = file.stem
        dfs[key] = df

    # Carga de anexos

    for file in sorted(ruta.glob("EPAAnexo_*")):
        if file.suffix == ".tab":
            df = pd.read_csv(file, sep="\t", encoding="utf-8", low_memory=False)
        else:
            df = pd.read_csv(file, encoding="utf-8", low_memory=False)

        key = file.stem
        dfs_a[key] = df

    return dfs, dfs_a

In [3]:
dfs, dfs_a = cargar_csv()

In [4]:
dfs["EPA_2021T1"]

,CICLO,CCAA,PROV,NVIVI,NIVEL,NPERS,EDAD1,RELPP1,SEXO1,NCONY,...,SIDI3,SIDAC1,SIDAC2,DAUSVAC,DAUSENF,DAUSOTR,TRAANT,AOI,RELLB4,FACTOREL
0,194,16,1,1,1,1,40,1,1,2,...,,1,1,2.0,NaN,NaN,1,04,,193.90
1,194,16,1,1,1,2,30,2,6,1,...,,1,1,NaN,NaN,NaN,1,03,,193.90
2,194,16,1,2,1,1,35,1,6,2,...,,1,1,1.0,NaN,NaN,1,04,,252.32
3,194,16,1,2,1,2,30,2,1,1,...,,1,1,1.0,NaN,NaN,1,04,,252.32
4,194,16,1,2,2,3,0,3,6,0,...,,,,NaN,NaN,NaN,,,,252.32
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
142845,194,52,52,57996,1,2,25,3,6,0,...,,1,1,NaN,NaN,NaN,6,04,,155.14
142846,194,52,52,57996,1,3,20,3,6,0,...,,2,2,NaN,NaN,NaN,6,05,,155.14
142847,194,52,52,57997,1,1,30,1,1,2,...,,1,1,NaN,NaN,NaN,1,04,,439.75
142848,194,52,52,57997,1,2,30,2,6,1,...,,1,1,NaN,NaN,NaN,1,04,,439.75


In [ ]:
# Crear la carpeta de salida si no existe
ruta_salida = Path("../data/interim")
ruta_salida.mkdir(parents=True, exist_ok=True)

for nombre, df in dfs.items():
    # Hacemos una copia para no tocar el original
    df_nuevo = df.rename(columns={"HORPLU": "HOREPLU"})
    
    # Guardamos el nuevo CSV
    df_nuevo.to_csv(ruta_salida / f"{nombre}.csv", index=False)

A continuación vamos a hacer otra función para cargar los nuevos csv que son los que vamos a mergear.

In [2]:
def cargar_csv_int():
    """
    Carga los archivos principales de la EPA y los anexos
    desde data/interim y devuelve dos diccionarios:
    - dfs: archivos principales
    - dfs_a: anexos
    """

    ruta = Path("../data/interim")  # carpeta donde están los datos

    dfs = {}    # aquí guardamos los df principales
    dfs_a = {}  # aquí guardamos los df de los anexos

    # Carga de archivos principales

    for file in sorted(ruta.glob("EPA_*")):
        if file.suffix == ".tab":
            df = pd.read_csv(file, sep="\t", encoding="utf-8", low_memory=False)
        else:
            df = pd.read_csv(file, sep="\t", encoding="utf-8", low_memory=False)
            # esto lo dejamos así porque, aunque la extensión sea distinta,
            # en la práctica también van separados por tabulación

        key = file.stem
        dfs[key] = df

    # Carga de anexos

    for file in sorted(ruta.glob("EPAAnexo_*")):
        if file.suffix == ".tab":
            df = pd.read_csv(file, sep="\t", encoding="utf-8", low_memory=False)
        else:
            df = pd.read_csv(file, encoding="utf-8", low_memory=False)

        key = file.stem
        dfs_a[key] = df

    return dfs, dfs_a

In [3]:
dfs, dfs_a = cargar_csv_int()

Aquí comprobamos, de nuevo, que todas las columnas se llaman igual

In [4]:
lista_columnas = []
for e in dfs.values():
    c= set(e.columns)
    lista_columnas.append(c)

i = 0
ref = lista_columnas[0]

for c in lista_columnas:
    i +=1
    if c == lista_columnas[0]:
        print(f"El grupo {i} es igual, todo bien.")
    else:
        faltan = ref - c
        sobran = c - ref
        print(f"En el grupo{i} faltan estas columnas: \n{faltan}\n y sobran estas columnas \n{sobran}\n")
              

El grupo 1 es igual, todo bien.
El grupo 2 es igual, todo bien.
El grupo 3 es igual, todo bien.
El grupo 4 es igual, todo bien.
El grupo 5 es igual, todo bien.
El grupo 6 es igual, todo bien.
El grupo 7 es igual, todo bien.
El grupo 8 es igual, todo bien.
El grupo 9 es igual, todo bien.
El grupo 10 es igual, todo bien.
El grupo 11 es igual, todo bien.
El grupo 12 es igual, todo bien.
El grupo 13 es igual, todo bien.
El grupo 14 es igual, todo bien.
El grupo 15 es igual, todo bien.
El grupo 16 es igual, todo bien.
El grupo 17 es igual, todo bien.
El grupo 18 es igual, todo bien.
El grupo 19 es igual, todo bien.
El grupo 20 es igual, todo bien.


In [5]:
ruta = Path("../data/interim")  # carpeta donde están los datos
archivos = glob.glob(os.path.join(ruta, "*.csv"))
dfs = [pd.read_csv(f) for f in archivos]
df_final = pd.concat(dfs, ignore_index=True)
df_final.to_csv("EPA_joined.csv", index=False)


In [ ]:
df_final